In [ ]:
# In this file, we test 1000 QTSA on finetuned Llama3.2-1B-Intruct model (with SFT)

In [ ]:
# top 300 QTSA test


import torch, os, re
import json
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

base_model_path = "Llama3.2-1B-Instruct"
lora_model_path = "outputs/(new)Llama3.2-1B-Instruct-qtsa-mix/checkpoint-5600"
tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    device_map="auto",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(base_model, lora_model_path)
model = model.merge_and_unload()
model.eval()

jsonl_path = "testbench/mcQTSA_test_final.jsonl"
output_path = "1000QTSA_test/(new)300_finetuned_1B_Instruct_prompt2.jsonl" 

print(f"\nstart to deal with the file: {jsonl_path}")

total_start_time = time.time()

results = []
max_items = 300
with open(jsonl_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if not line.strip():
            continue
            
        if i >= max_items:
            break

        try:
            data = json.loads(line.strip())
            question = data.get("question", "")
            correct_answer = data.get("answer", "")
            id_num = data.get("id", i + 1)

            print(f"\nSolving {i+1} data (ID: {id_num}): {question[:50]}...")

            prompt = (
                "<|begin_of_text|>"
                "<|start_header_id|>user<|end_header_id|>\n"
                "You are a Radio Frequency expert.\n"
                "Please answer questions using concise and professional language, "
                "providing the core content directly without excessive explanation.\n\n"
                f"Question:\n{question}\n"
                "<|eot_id|>"
                "<|start_header_id|>assistant<|end_header_id|>\n"
            )

            start_time = time.time()
            
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            input_tokens_count = inputs.input_ids.shape[1]
            print(f"Input tokens: {input_tokens_count}")
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=16000,  
                    repetition_penalty=1.2,
                    do_sample=False,
                    #top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            output_tokens_count = outputs[0].shape[0] - inputs.input_ids.shape[1]
            
            full_response = tokenizer.decode(
                outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
            )

            print(f"Output tokens: {output_tokens_count}")
            
            result = {
                "id": id_num,
                "question": question,
                "output": full_response,
                "correct_answer": correct_answer,
            }
            results.append(result)

        except json.JSONDecodeError as e:
            print(f"JSON parsing error on line {i+1}: {e}")
        except Exception as e:
            print(f"Error occurred while processing line {i+1}: {e}")

total_end_time = time.time()
total_processing_time = total_end_time - total_start_time

if results:
    avg_time_per_question = total_processing_time / len(results)

    #print(f"total question: {len(results)}")
    print(f"total time cost: {total_processing_time:.2f}s")
    print(f"average time per question: {avg_time_per_question:.2f}s")
    print(f"================\n")
    
with open(output_path, "w", encoding="utf-8") as f:
    for result in results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

print(f"\nFinish！Output is saved in: {output_path}")
print(f"solve {len(results)} datas")

In [ ]:
# 1000 QTSA test

import torch, os, re
import json
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

base_model_path = "Llama3.2-1B-Instruct"
lora_model_path = "outputs/(new)Llama3.2-1B-Instruct-qtsa-mix/checkpoint-5600"
tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    device_map="auto",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(base_model, lora_model_path)
model = model.merge_and_unload()
model.eval()

jsonl_path = "testbench/mcQTSA_test_final.jsonl"
output_path_300 = "1000QTSA_test/(new)300_finetuned_1B_Instruct_prompt2.jsonl" 
output_path_1000 = "1000QTSA_test/(new)1000_finetuned_1B_Instruct_prompt2.jsonl" 

print(f"\nstart to deal with the file: {jsonl_path}")

existing_results = []
if os.path.exists(output_path_300):
    with open(output_path_300, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                existing_results.append(json.loads(line.strip()))
    print(f"Loaded {len(existing_results)} existing results from previous run")

total_start_time = time.time()

results = existing_results.copy()  
start_index = 300 
max_items = 1000   

with open(jsonl_path, 'r', encoding='utf-8') as f:
    all_lines = f.readlines()
    
    for i in range(start_index, min(max_items, len(all_lines))):
        line = all_lines[i]
        if not line.strip():
            continue

        try:
            data = json.loads(line.strip())
            question = data.get("question", "")
            correct_answer = data.get("answer", "")
            id_num = data.get("id", i + 1)

            print(f"\nSolving {i+1} data (ID: {id_num}): {question[:50]}...")

            prompt = (
                "<|begin_of_text|>"
                "<|start_header_id|>user<|end_header_id|>\n"
                "You are a Radio Frequency expert.\n"
                "Please answer questions using concise and professional language, "
                "providing the core content directly without excessive explanation.\n\n"
                f"Question:\n{question}\n"
                "<|eot_id|>"
                "<|start_header_id|>assistant<|end_header_id|>\n"
            )

            start_time = time.time()
            
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            input_tokens_count = inputs.input_ids.shape[1]
            print(f"Input tokens: {input_tokens_count}")
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=16000,  
                    repetition_penalty=1.2,
                    do_sample=False,
                    #top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            output_tokens_count = outputs[0].shape[0] - inputs.input_ids.shape[1]
            
            full_response = tokenizer.decode(
                outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
            )

            print(f"Output tokens: {output_tokens_count}")
            
            result = {
                "id": id_num,
                "question": question,
                "output": full_response,
                "correct_answer": correct_answer,
            }
            results.append(result)

        except json.JSONDecodeError as e:
            print(f"JSON parsing error on line {i+1}: {e}")
        except Exception as e:
            print(f"Error occurred while processing line {i+1}: {e}")

total_end_time = time.time()
total_processing_time = total_end_time - total_start_time

if results:
    avg_time_per_question = total_processing_time / (len(results) - len(existing_results))

    print(f"total time cost: {total_processing_time:.2f}s")
    print(f"average time per question: {avg_time_per_question:.2f}s")
    print(f"================\n")

with open(output_path_1000, "w", encoding="utf-8") as f:
    for result in results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

print(f"\nFinish！Output is saved in: {output_path_1000}")